# Pauli-Basis Superoperators

This notebook fills a gap in the examples: it shows how `EDKit.jl` represents operators and superoperators in the Pauli basis. The key functions are `pauli`, `pauli_list`, `commutation_mat`, and `dissipation_mat`.

The goal is to make the translation between dense matrix formulas and Pauli-coordinate linear algebra completely explicit.

In [ ]:
import Pkg
using LinearAlgebra

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
    include(joinpath(REPO_ROOT, "src", "EDKit.jl"))
    using .EDKit
else
    using EDKit
end


## A density matrix in Pauli coordinates

We start with a one-qubit density matrix. `pauli_list(ρ)` converts the matrix into its Pauli-basis coefficient vector, while `pauli(coeffs)` converts back to the matrix form.

In [ ]:
rho = ComplexF64[0.7 0.2 - 0.1im; 0.2 + 0.1im 0.3]
coeffs = pauli_list(rho)
rho_back = pauli(coeffs)

summary_roundtrip = (
    coefficient_vector = coeffs,
    roundtrip_error = norm(rho - rho_back),
)
summary_roundtrip


## Hamiltonian commutator as a matrix acting on Pauli coefficients

For a Hamiltonian `H`, the Liouvillian piece `-i[H, ρ]` is linear in `ρ`. In the Pauli basis, that becomes an ordinary matrix multiplication by `commutation_mat(H)`.

In [ ]:
H = 0.8 * spin("X") + 0.4 * spin("Z")
comm = commutation_mat(H)

lhs = pauli(comm * coeffs)
rhs = -1im * (H * rho - rho * H)

summary_commutator = (
    pauli_space_error = norm(lhs - rhs),
    commutator_matrix_size = size(comm),
)
summary_commutator


## Lindblad dissipator in Pauli coordinates

The dissipative term generated by one jump operator `L` can be written as

$$\mathcal{D}_L(\rho) = L \rho L^\dagger - \tfrac{1}{2}(L^\dagger L \rho + \rho L^\dagger L).$$

`dissipation_mat(L)` gives the corresponding matrix in Pauli coordinates.

In [ ]:
jump = sqrt(0.3) * ComplexF64[0 1; 0 0]
diss = dissipation_mat(jump)

lhs_diss = pauli(diss * coeffs)
rhs_diss = jump * rho * jump' - (jump' * jump * rho + rho * jump' * jump) / 2

summary_dissipator = (
    pauli_space_error = norm(lhs_diss - rhs_diss),
    dissipator_matrix_size = size(diss),
)
summary_dissipator


## Combine coherent and dissipative evolution

Once both pieces are in Pauli coordinates, we can simply add them. This is the main practical reason to work in the Pauli basis: superoperators become ordinary matrices on coefficient vectors.

In [ ]:
liouvillian = comm + diss
pauli_rhs = pauli(liouvillian * coeffs)
dense_rhs = -1im * (H * rho - rho * H) + jump * rho * jump' - (jump' * jump * rho + rho * jump' * jump) / 2

summary_liouvillian = (
    combined_error = norm(pauli_rhs - dense_rhs),
    eigenvalues = eigvals(liouvillian),
)
summary_liouvillian


## Plot the Pauli coefficients

A simple bar plot is enough to visualize the Pauli-coordinate representation. This is especially useful once you move beyond one qubit and want to see which operator components are carrying the weight.

In [ ]:
labels = ["I", "X", "Y", "Z"]
values = real.(coeffs)

if isdefined(Main, :IJulia)
    using Plots
    bar(labels, values; xlabel = "Pauli basis element", ylabel = "coefficient", label = nothing, title = "Real part of Pauli coefficients")
else
    @info "Skipping Plots outside IJulia; returning coefficients instead"
    (; labels = labels, values = values)
end


## Takeaway

This notebook is the missing bridge between the dense Lindblad formulas and the operator-space utilities elsewhere in the package. If you want to understand how `EDKit.jl` moves from matrices to superoperators, this is the place to start before going on to the Lindblad and tensor-network examples.